In [7]:
!nvcc --version
!nvidia-smi --query-gpu=name,driver_version --format=csv,noheader

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Tue_Oct_29_23:50:19_PDT_2024
Cuda compilation tools, release 12.6, V12.6.85
Build cuda_12.6.r12.6/compiler.35059454_0
Tesla T4, 560.35.03


In [ ]:
!pip install ninja pytest
!git clone https://github.com/gkienpham-cmd/flashattention-cuda
%cd flashattention-cuda
# base image ships a CUDA-13 torch that won't init on a 12.x driver -> install the cu124 build (works on any T4 host)
!pip install --force-reinstall --no-cache-dir torch --index-url https://download.pytorch.org/whl/cu124
!ln -sf $(which python3) /usr/local/bin/python


In [9]:
!python -c "import torch; print('cuda_ok', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))"
!ls kernels/    # expect: v1_naive  v2_tiled  v3_online

cuda_ok True | Tesla T4
v1_naive  v2_tiled  v3_online


In [11]:
%%writefile mem_check.py
import torch
from fa_kernels import attention
B,H,N,d = 1,8,8192,64
for backend in ["v2_tiled", "v3_online"]:
    q=torch.randn(B,H,N,d,device='cuda'); k=torch.randn(B,H,N,d,device='cuda'); v=torch.randn(B,H,N,d,device='cuda')
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); base=torch.cuda.memory_allocated()
    out=attention(q,k,v,backend=backend); torch.cuda.synchronize()
    print(f"{backend}: peak +{(torch.cuda.max_memory_allocated()-base)/1e6:.1f} MB  (a materialized S = {B*H*N*N*4/1e6:.0f} MB)")
    del q,k,v,out; torch.cuda.empty_cache()


Writing mem_check.py


In [12]:
!python mem_check.py

Using /root/.cache/torch_extensions/py312_cu124 as PyTorch extensions root...
Creating extension directory /root/.cache/torch_extensions/py312_cu124/fa_v2_tiled...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py312_cu124/fa_v2_tiled/build.ninja...
/usr/local/lib/python3.12/dist-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module fa_v2_tiled...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)
[1/3] c++ -MMD -MF binding.o.d -DTORCH_EXTENSION_NAME=fa_v2_tiled -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.12/dist-packages/torch/include -isystem /usr/l

In [15]:
%%writefile prof_check.py
import torch
from torch.profiler import profile, ProfilerActivity
from fa_kernels import attention
B,H,N,d = 1,8,8192,64
q=torch.randn(B,H,N,d,device='cuda'); k=torch.randn(B,H,N,d,device='cuda'); v=torch.randn(B,H,N,d,device='cuda')
for _ in range(3): attention(q,k,v,backend="v3_online")   # warmup + JIT
torch.cuda.synchronize()
with profile(activities=[ProfilerActivity.CUDA]) as prof:
    for _ in range(10): attention(q,k,v,backend="v3_online")
    torch.cuda.synchronize()
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))

Overwriting prof_check.py


In [16]:
!python prof_check.py

Using /root/.cache/torch_extensions/py312_cu124 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py312_cu124/fa_v3_online/build.ninja...
/usr/local/lib/python3.12/dist-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module fa_v3_online...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)
ninja: no work to do.
Loading extension module fa_v3_online...
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     C

In [ ]:
!python -m pytest tests/test_correctness.py -k v3_online -v

============================= test session starts ==============================
platform linux -- Python 3.12.3, pytest-9.1.0, pluggy-1.6.0 -- /usr/local/bin/python
cachedir: .pytest_cache
rootdir: /flashattention-cuda/flashattention-cuda
configfile: pyproject.toml
plugins: anyio-4.13.0
collected 41 items / 26 deselected / 15 selected

tests/test_correctness.py::test_matches_sdpa[False-1-4-128-64-v3_online] PASSED [  6%]
tests/test_correctness.py::test_matches_sdpa[False-2-8-512-64-v3_online] PASSED [ 13%]
tests/test_correctness.py::test_matches_sdpa[False-1-8-512-128-v3_online] PASSED [ 20%]
tests/test_correctness.py::test_matches_sdpa[False-1-2-2048-64-v3_online] PASSED [ 26%]
tests/test_correctness.py::test_matches_sdpa[False-1-2-130-64-v3_online] PASSED [ 33%]
tests/test_correctness.py::test_matches_sdpa[False-1-2-100-128-v3_online] PASSED [ 40%]
tests/test_correctness.py::test_matches_sdpa[True-1-4-128-64-v3_online] PASSED [ 46%]
tests/test_correctness.py::test_matches_sdpa[True-